In [1]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.uncertainty_propagation_t as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Hungerford RI23 events #
####################

Hungerford_tracers = ['Ca_mg_L', 'Cl_mg_L', 'Si_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']

# 1. Run EMMA function
(
    hungerford_febros_fractions_df,
    hungerford_febros_scaler,
    hungerford_febros_pca,
    hungerford_febros_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Hungerford",
    start_date="2023-02-08 00:00:00",
    end_date="2023-02-12 00:00:00",
    endmember_ids=[
                   "RI23-1001", # Baseflow 02/09/2023
                   "RI23-5003", # SWLD 02/15/2023
                   "RI23-5002" #  SML 02/15/2023
                    ], 
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-1001", "RI23-5003", "RI23-5002"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Hungerford")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-02-08 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-02-12 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Cl_mg_L": 0.15,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# 5. Calculate Genereux Uncertainties
uncertainty_df = up.propagate_genereux_uncertainty(
    stream_df=stream_event_df,
    em_grouped=hungerford_febros_endmembers_df,
    em_raw=em_raw_subset,
    tracers=Hungerford_tracers,
    analytical_sd=analytical_sd,
    confidence_level=0.95
)

# 6. Merge fractions and their calculated uncertainties for a complete table
results_with_error = pd.merge(
    hungerford_febros_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
)

# Identify which uncertainty columns were actually generated
uncertainty_cols = [
    col for col in results_with_error.columns if "Uncertainty_1sig" in col
]
# Extract the base fraction names (e.g., "Groundwater", "Snowmelt lysimeter")
fraction_cols = [col.replace("_Uncertainty_1sig", "") for col in uncertainty_cols]

# Combine them in alternating order: [Fraction_1, Uncertainty_1, Fraction_2, Uncertainty_2...]
display_cols = ["Sample ID", "Datetime"]
for frac, unc in zip(fraction_cols, uncertainty_cols):
    display_cols.extend([frac, unc])

# Print the head of the dynamically built column list
print(results_with_error[display_cols].head())
results_with_error.head(20)

   Sample ID            Datetime
0  RI23-1005 2023-02-10 10:00:00
1  RI23-1007 2023-02-10 16:00:00
2  RI23-1008 2023-02-11 12:30:00
3  RI23-1002 2023-02-09 16:00:00
4  RI23-1003 2023-02-09 22:00:00


,Sample ID,Datetime,Site,Groundwater,Snowmelt lysimeter,Soil water lysimeter,Sum_Fractions,Groundwater_Uncertainty_95sig,Snowmelt lysimeter_Uncertainty_95sig,Soil water lysimeter_Uncertainty_95sig
0,RI23-1005,2023-02-10 10:00:00,Hungerford,0.145442,9.640563e-15,8.545576e-01,1.0,2.742405,2.742405e+00,3.507713e-07
1,RI23-1007,2023-02-10 16:00:00,Hungerford,0.393252,-1.182702e-15,6.067480e-01,1.0,1.694802,2.498535e+01,2.442710e+01
2,RI23-1008,2023-02-11 12:30:00,Hungerford,1.000000,3.885781e-16,3.164136e-15,1.0,1.937141,1.827733e+01,1.785753e+01
3,RI23-1002,2023-02-09 16:00:00,Hungerford,0.957676,-1.928295e-16,4.232444e-02,1.0,0.000507,2.672211e-04,2.399861e-04
4,RI23-1003,2023-02-09 22:00:00,Hungerford,NaN,NaN,NaN,0.0,3.649435,9.036609e-09,3.649435e+00


In [ ]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.uncertainty_propagation_t as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Hungerford RI23 events #
####################

Hungerford_tracers = ['Ca_mg_L', 'Cl_mg_L', 'Si_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']

# 1. Run EMMA function
(
    hungerford_martherm_fractions_df,
    hungerford_martherm_scaler,
    hungerford_martherm_pca,
    hungerford_martherm_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Hungerford",
    start_date="2023-03-21 00:00:00",
    end_date="2023-03-26 00:00:00",
    endmember_ids=[
                   "RI23-1001", # Baseflow 02/09/2023 (labeled GW)
                   "RI23-5008", # Mark's well 2023
                   "RI23-5007", # SWLD
                   "RI23-1061"
                    ], 
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin([["RI23-1001", "RI23-5008", "RI23-5007", "RI23-1061"]])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Hungerford")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-03-21 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-03-26 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Cl_mg_L": 0.15,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# 5. Calculate Genereux Uncertainties
uncertainty_df = up.propagate_genereux_uncertainty(
    stream_df=stream_event_df,
    em_grouped=hungerford_martherm_endmembers_df,
    em_raw=em_raw_subset,
    tracers=Hungerford_tracers,
    analytical_sd=analytical_sd,
    confidence_level=0.95
)

# 6. Merge fractions and their calculated uncertainties for a complete table
results_with_error = pd.merge(
    hungerford_martherm_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
)

# Identify which uncertainty columns were actually generated
uncertainty_cols = [
    col for col in results_with_error.columns if "Uncertainty_1sig" in col
]
# Extract the base fraction names (e.g., "Groundwater", "Snowmelt lysimeter")
fraction_cols = [col.replace("_Uncertainty_1sig", "") for col in uncertainty_cols]

# Combine them in alternating order: [Fraction_1, Uncertainty_1, Fraction_2, Uncertainty_2...]
display_cols = ["Sample ID", "Datetime"]
for frac, unc in zip(fraction_cols, uncertainty_cols):
    display_cols.extend([frac, unc])

# Print the head of the dynamically built column list
print(results_with_error[display_cols].head())
results_with_error.head(20)

In [ ]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.uncertainty_propagation_t as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Hungerford RI23 events #
####################

hungerford_tracers = ['Ca_mg_L', 'Cl_mg_L', 'Si_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']

# 1. Run EMMA function
(
    hungerford_fmelt_fractions_df,
    hungerford_fmelt_scaler,
    hungerford_fmelt_pca,
    hungerford_fmelt_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Hungerford",
    start_date="2023-03-30 00:00:00",
    end_date="2023-04-04 00:00:00",
    endmember_ids=[
                   "RI23-1001", # Baseflow 02/09/2023 (labeled GW)
                   "RI23-5008", # Groundwater (Mark's well) 03/16/2023
                   "RI23-5013", # Soil water lysimeter wet 04/12/23
                   "RI23-5017", # Snowmelt lysimeter 04/12/23
                   "RI23-1061" # Snowmelt lysimeter 03/28/23
                    ],
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-1001", "RI23-5008", "RI23-5013", "RI23-5017", "RI23-1061"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Hungerford")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-03-30 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-04-04 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Cl_mg_L": 0.15,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# 5. Calculate Genereux Uncertainties
uncertainty_df = up.propagate_genereux_uncertainty(
    stream_df=stream_event_df,
    em_grouped=hungerford_fmelt_endmembers_df,
    em_raw=em_raw_subset,
    tracers=hungerford_tracers,
    analytical_sd=analytical_sd,
    confidence_level=0.95
)

# 6. Merge fractions and their calculated uncertainties for a complete table
results_with_error = pd.merge(
    hungerford_fmelt_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
)

# Identify which uncertainty columns were actually generated
uncertainty_cols = [
    col for col in results_with_error.columns if "Uncertainty_1sig" in col
]
# Extract the base fraction names (e.g., "Groundwater", "Snowmelt lysimeter")
fraction_cols = [col.replace("_Uncertainty_1sig", "") for col in uncertainty_cols]

# Combine them in alternating order: [Fraction_1, Uncertainty_1, Fraction_2, Uncertainty_2...]
display_cols = ["Sample ID", "Datetime"]
for frac, unc in zip(fraction_cols, uncertainty_cols):
    display_cols.extend([frac, unc])

# Print the head of the dynamically built column list
print(results_with_error[display_cols].head())
results_with_error.head(20)